# InstaQuery - model comparison study (Colab T4)
Compares 8 open models (0.5B-7B) with the QLoRA fine-tuned Qwen2.5-Coder-1.5B on Spider dev, the quick-commerce set
and a 500-question stratified sample of **BIRD dev**, and tests two fixes: **skeleton RAG** and **execution-guided self-correction**.
The fine-tuned rows need the LoRA adapter written by `InstaQuery_Colab.ipynb` (`MyDrive/InstaQuery/outputs_full/model/lora_adapter`).
Every run is cached in Drive, so after a disconnect just run all cells again. Results: `experiments/README.md`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/InstaQuery"
BIRD_N = 500
import os, subprocess, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q -U transformers peft accelerate bitsandbytes datasets sentence-transformers sqlglot sqlparse nltk gdown huggingface_hub httpx nest_asyncio
!pip uninstall -y -q torchao
!rm -rf /content/instaquery && git clone -q https://github.com/tarubee19/instaquery /content/instaquery && mkdir -p /content/instaquery/experiments

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, "/content/instaquery/src"); sys.path.insert(0, "/content/instaquery/experiments")
from instaquery.config import load_config
from instaquery.pipeline import Workspace
from instaquery.serving import install_vllm
from instaquery.official_eval import ensure_repo, add_hardness
from compare_models import Study, MODELS
cfg = load_config(run_mode="full", output_dir=f"{DRIVE_DIR}/outputs_full", vllm_max_model_len=8192, vllm_gpu_mem=0.88)
ws = Workspace(cfg, Path("/content/instaquery/data/quickcommerce_eval.json")).prepare_data()
OFFICIAL = ensure_repo(Path(cfg.work_dir) / "test-suite-sql-eval")
add_hardness(ws.dev, OFFICIAL, ws.db_lookup)
assert install_vllm(cfg.vllm_version, log_file=Path("/content/vllm_install.log"))
ws.backend = "vllm"
study = Study(ws, cfg.output_dir, Path(cfg.work_dir) / "bird", BIRD_N)
print("BIRD:", len(study.examples("bird_dev")))

In [ ]:
# fine-tuned model (rebuilt from the saved LoRA adapter)
from instaquery.train import merge_adapter
merged = str(merge_adapter(cfg.base_model, cfg.adapter_dir, cfg.merged_dir))

In [ ]:
def run_model(name, path, params, extra, plan):
    todo = [p for p in plan if not (study.out / "metrics" / f"{p[0]}__{name}__{p[1]}{'__repair' if p[2] else ''}.json").exists()]
    if not todo:
        print("[done]", name); return
    try:
        with ws.generator(path, backend="vllm", tag=name.replace('.', '_')) as gen:
            for d, shots, rep in todo:
                study.run(gen, name, d, shots, rep, extra, params)
    except Exception as e:
        print("FAILED", name, repr(e)[:500])
FULL = [("spider_dev", "zero", False), ("spider_dev", "skeleton", False), ("spider_dev", "zero", True),
        ("quickcommerce", "zero", False), ("quickcommerce", "zero", True), ("bird_dev", "zero", False), ("bird_dev", "zero", True)]
run_model("finetuned-1.5b", merged, 1.5, {}, FULL)
run_model("qwen2.5-coder-1.5b", *MODELS["qwen2.5-coder-1.5b"], FULL)

In [ ]:
OTHERS = [("spider_dev", "zero", False), ("quickcommerce", "zero", False), ("bird_dev", "zero", False)]
for name in ["qwen2.5-coder-0.5b", "deepseek-coder-1.3b", "qwen3-1.7b", "llama-3.2-3b", "qwen2.5-coder-3b", "qwen3-4b", "qwen2.5-coder-7b-awq"]:
    run_model(name, *MODELS[name], OTHERS)
    print(f"elapsed {(time.time()-T0)/60:.0f} min")

In [ ]:
import shutil
study.summarize()
for m in json.loads((study.out / "summary.json").read_text()):
    print(f"{m['dataset']:13s} {m['model']:22s} {m['shots']:8s} {'repair' if m['repair'] else '      '} EX={100*m['execution_accuracy']:.1f}% valid={100*m['valid_sql_rate']:.1f}% n={m['n']} p={m.get('vs_reference',{}).get('p_value','-')}")
shutil.make_archive(f"{DRIVE_DIR}/comparison_results", "zip", study.out)
print("ZIP SAVED", f"{DRIVE_DIR}/comparison_results.zip", f"total {(time.time()-T0)/60:.0f} min")